# 01 · Data ingest & EDA
**Dataset:** [Open Food Facts (with Nutriscore & Generic Names)](https://www.kaggle.com/datasets/paufortiana/open-food-facts-with-nutriscore-and-generic-names)

**Phase 1 (tabular):** load nutrition data, EDA, clean.  
**Phase 2 (corpus):** fetch nutrition text and chunk it for the retriever.


In [ ]:
# Make src/ importable whether on Colab or local
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[0] if (pathlib.Path.cwd().name=='notebooks') else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
from nutricoach import config as C


## 1. Download the dataset
`kagglehub` needs Kaggle credentials: on Colab upload `kaggle.json`, or set `KAGGLE_USERNAME` / `KAGGLE_KEY`. Or download the CSV manually into `data/raw/`.


In [ ]:
from nutricoach import data
csv_path = data.download_raw()   # returns the CSV path (or drop the file in data/raw/)
print('CSV:', csv_path)


## 2. Load & clean
`load_clean()` maps OFF's raw column names to our canonical schema, coerces numerics, drops rows without a calorie target, clips impossible values, and fills undeclared nutrients with 0.


In [ ]:
df = data.load_clean()
print(df.shape)
df.head()


In [ ]:
df.describe()


## 3. EDA — regression target (calories)


In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.histplot(df[C.REGRESSION_TARGET], bins=60); plt.title('kcal / 100g'); plt.show()


## 4. EDA — Nutri-Score class balance
Expect imbalance across A–E; note it for the classification task (class weights / SMOTE).


In [ ]:
df[C.CLASSIFICATION_TARGET].value_counts(dropna=False)


In [ ]:
# correlation of nutrients with calories (sanity: fat/carbs should correlate up)
df[C.NUTRIENT_FEATURES + [C.REGRESSION_TARGET]].corr()[C.REGRESSION_TARGET].sort_values()


## 5. Save the processed frame
So notebooks 02/03 don't re-clean from scratch.


In [ ]:
df.to_parquet(C.PROCESSED_DIR / 'food_clean.parquet', index=False)
print('saved ->', C.PROCESSED_DIR / 'food_clean.parquet')


---
## Phase 2 — build the knowledge corpus (do later, before notebook 04)
Open sources: Wikipedia (nutrition/supplements), OpenStax *Nutrition* (CC BY), WHO/USDA guidelines. Keep it to a few hundred–few thousand chunks.


In [ ]:
from nutricoach import ingest
documents = [
    # {'text': open('data/raw/openstax_nutrition.txt').read(), 'source': 'OpenStax Nutrition'},
    # {'text': wiki_text, 'source': 'Wikipedia: Protein'},
]
chunks = ingest.build_corpus(documents)
print(len(chunks), 'chunks')
